# 12. AutoGaze Full Pipeline 추론 가이드

> **AutoGaze + ViT + MLLM** 전체 파이프라인을 인터랙티브하게 탐색합니다.  
> `infer.py` (AutoGaze 단독) 와 `infer_full.py` (전체 QA) 모두 다룹니다.

## 목차
1. 환경 설정
2. MLLM 백엔드 개요
3. AutoGaze 단독 — gaze map 추출 (`infer.py` 방식)
4. NVILA 추론 — AutoGaze ON/OFF 비교
5. Qwen2.5-VL 추론 — hook vs full 통합
6. V-JEPA2 — 특징 추출 & LLM 추론
7. Gazing Ratio Sweep
8. 타이밍 분석
9. 전체 CLI 가이드

---
## 1. 환경 설정

In [ ]:
import sys, os, warnings, time
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

import torch
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import torch.nn.functional as F
from pathlib import Path
from PIL import Image

# ── 한국어 폰트 ────────────────────────────────────────────────────────────
def _configure_mpl_cjk():
    matplotlib.rcParams["axes.unicode_minus"] = False
    try:
        import matplotlib.font_manager as fm
        for name in ["Apple SD Gothic Neo", "AppleGothic", "NanumGothic",
                     "Noto Sans CJK KR", "DejaVu Sans"]:
            try:
                path = fm.findfont(fm.FontProperties(family=name), fallback_to_default=False)
                if path and os.path.exists(path):
                    matplotlib.rcParams["font.family"] = name
                    return name
            except Exception:
                pass
    except Exception:
        pass
    return "default"

font_name = _configure_mpl_cjk()

# ── 디바이스 ───────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

def _sync():
    if device.type == "cuda":  torch.cuda.synchronize()
    elif device.type == "mps": torch.mps.synchronize()

print(f"device : {device}")
print(f"font   : {font_name}")

In [ ]:
# ── 경로 설정 ──────────────────────────────────────────────────────────────
REPO_ROOT        = Path("..").resolve()
MODEL_PATH       = REPO_ROOT / "weights" / "NVILA-8B-HD-Video"
AG_PATH          = REPO_ROOT / "weights" / "AutoGaze"
VIDEO_PATH       = REPO_ROOT / "assets" / "example_input.mp4"
QWEN_PATH        = REPO_ROOT / "weights" / "Qwen2.5-VL-7B-Instruct"  # bash scripts/download_models.sh weights qwen25vl
VJEPA2_PATH      = REPO_ROOT / "weights" / "vjepa2-vitl-fpc64-256"    # bash scripts/download_models.sh weights vjepa2
LM_PATH          = REPO_ROOT / "weights" / "Qwen2.5-7B-Instruct"      # [vjepa2_llm] bash scripts/download_models.sh weights qwen25
PROJECTOR_PATH   = None                              # [vjepa2_llm] None = random init

# ── 상태 확인 ──────────────────────────────────────────────────────────────
def _check(path, label):
    p = Path(path)
    status = "✓" if p.exists() else "✗ 없음"
    print(f"  {label:22} {status}  ({p})")
    return p.exists()

print("가중치 파일 확인:")
has_nvila = _check(MODEL_PATH, "NVILA-8B-HD-Video")
has_ag    = _check(AG_PATH, "AutoGaze")
_check(VIDEO_PATH, "example_input.mp4")

RATIO = 0.75
NUM_FRAMES = 16

print()
print(f"Gazing ratio : {RATIO}")
print(f"Frames       : {NUM_FRAMES}")

---
## 2. MLLM 백엔드 개요

In [ ]:
from autogaze.eval.models import RUNNERS

print("지원 MLLM 백엔드")
print("─" * 90)
info = {
    "nvila":          ("NVILA-8B-HD-Video",   "SigLIP-L/14",  "NVILAProcessor 내장",  "✓"),
    "qwen25vl":       ("Qwen2.5-VL-7B",       "Qwen ViT",     "zero-shot hook",        "✓"),
    "qwen25vl_full":  ("Qwen2.5-VL-7B",       "Qwen ViT",     "class monkey-patch",   "✓"),
    "vjepa2":         ("V-JEPA2 ViT-L",       "V-JEPA2 ViT",  "zero-shot hook",        "특징만"),
    "vjepa2_full":    ("V-JEPA2 ViT-L",       "V-JEPA2 ViT",  "class monkey-patch",   "특징만"),
    "vjepa2_llm":     ("V-JEPA2+projector+LM","V-JEPA2 ViT",  "class monkey-patch",   "✓"),
}
print(f"  {'키':18} {'모델':26} {'ViT':16} {'AutoGaze 통합':22} {'MCQ'}")
print("─" * 90)
for k, (model, vit, ag, mcq) in info.items():
    reg = "✓" if k in RUNNERS else "✗"
    print(f"  {k:18} {model:26} {vit:16} {ag:22} {mcq}  (등록:{reg})")

---
## 3. AutoGaze 단독 — Gaze Map 추출

In [ ]:
# CLI 사용법 확인
print("""
infer.py CLI 예시
─────────────────────────────────────────────────────────────────
# gaze map + 시각화 MP4
python -m autogaze.infer assets/example_input.mp4 --output-format video

# 모든 포맷 (json + viz + npy + video)
python -m autogaze.infer assets/example_input.mp4

# 모든 프레임 처리 (16프레임 청크 분할)
python -m autogaze.infer assets/example_input.mp4 --all-frames --output-format frames,video

# stride 샘플링
python -m autogaze.infer assets/example_input.mp4 --stride 10 --output-format frames,video
─────────────────────────────────────────────────────────────────
""")

In [ ]:
# AutoGaze 단독 실행 (infer.py 방식 — Python API)
import cv2
from autogaze.models.autogaze import AutoGaze, AutoGazeImageProcessor

assert has_ag, f"AutoGaze 가중치 없음: {AG_PATH}\n→ bash scripts/download_models.sh weights autogaze"

print("AutoGaze 로드 중...")
t0 = time.perf_counter()
ag_proc  = AutoGazeImageProcessor.from_pretrained(str(AG_PATH))
ag_model = AutoGaze.from_pretrained(str(AG_PATH)).to(device).eval()
print(f"  완료 ({time.perf_counter()-t0:.1f}s)")
print(f"  파라미터: {sum(p.numel() for p in ag_model.parameters()):,}")

In [ ]:
# 비디오 프레임 로드
def load_frames_uniform(video_path, num_frames=16):
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or num_frames
    indices = np.linspace(0, total-1, num=num_frames, dtype=int)
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()
        if ok:
            frames.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
        elif frames:
            frames.append(frames[-1])
    cap.release()
    while len(frames) < num_frames:
        frames.append(frames[-1])
    return frames[:num_frames]

frames = load_frames_uniform(VIDEO_PATH, NUM_FRAMES)
print(f"로드 완료: {len(frames)}프레임  ({frames[0].size[0]}×{frames[0].size[1]} px)")

In [ ]:
# Gaze map 계산
def get_gaze_map(frames, ratio=0.75):
    """returns gaze_map (T, 14, 14) numpy float32"""
    batch = ag_proc(images=frames, return_tensors="pt")["pixel_values"]
    batch = batch.to(device)   # already (1, T, 3, 224, 224)
    with torch.no_grad():
        out = ag_model({"video": batch}, gazing_ratio=ratio)
    # gazing_mask[-1]: finest 14×14 scale (1, T, 196)
    mask = out["gazing_mask"][-1][0].float()   # (T, 196)
    return mask.reshape(-1, 14, 14).cpu().numpy()

_sync(); t0 = time.perf_counter()
gaze_maps = get_gaze_map(frames, ratio=RATIO)
_sync(); t_ag = time.perf_counter() - t0

print(f"gaze_maps: {gaze_maps.shape}")
print(f"평균 선택 비율: {gaze_maps.mean():.2%}")
print(f"추론 시간: {t_ag*1000:.1f} ms")

In [ ]:
# Gaze Overlay 시각화
N_SHOW = min(8, len(frames))
fig, axes = plt.subplots(2, N_SHOW, figsize=(N_SHOW * 2.3, 5))

for i in range(N_SHOW):
    # 원본 프레임
    axes[0, i].imshow(frames[i])
    axes[0, i].axis("off")
    axes[0, i].set_title(f"F{i}", fontsize=8)

    # Gaze overlay
    gm = gaze_maps[i]           # (14, 14)
    frame_np = np.array(frames[i].resize((224, 224)))
    mask_up  = F.interpolate(
        torch.tensor(gm).unsqueeze(0).unsqueeze(0).float(),
        size=(224, 224), mode="nearest"
    ).squeeze().numpy()

    overlay = frame_np.copy().astype(float)
    overlay[mask_up == 0] *= 0.25
    axes[1, i].imshow(overlay.clip(0, 255).astype(np.uint8))
    axes[1, i].imshow(gm, cmap="cool", alpha=0.35,
                      extent=[0, 224, 224, 0], aspect="auto")
    axes[1, i].set_title(f"Gaze({int(gm.sum())}/196)", fontsize=8)
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("Original", fontsize=9)
axes[1, 0].set_ylabel(f"Gaze (r={RATIO})", fontsize=9)
fig.suptitle("AutoGaze 14×14 Gaze Map Overlay", fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

---
## 4. NVILA 추론 — AutoGaze ON/OFF 비교

In [ ]:
# ── Runner 레지스트리로 NVILA 로드 ────────────────────────────────────────
from autogaze.eval.models import load_runner

assert has_nvila, f"NVILA 가중치 없음: {MODEL_PATH}\n→ bash scripts/download_models.sh weights nvila"

print("[1/2] NVILA + AutoGaze ON 로드 중...")
t0 = time.perf_counter()
runner_ag = load_runner(
    mllm          = "nvila",
    model_path    = str(MODEL_PATH),
    autogaze_path = str(AG_PATH),
    gazing_ratio  = RATIO,
)
print(f"  완료 ({time.perf_counter()-t0:.1f}s)")

print("[2/2] NVILA AutoGaze OFF (기준선) 로드 중...")
t0 = time.perf_counter()
runner_base = load_runner(
    mllm          = "nvila",
    model_path    = str(MODEL_PATH),
    autogaze_path = str(AG_PATH),  # ratio=1.0 → 전체 패치 (AutoGaze OFF 동등)
    gazing_ratio  = 1.0,
)
print(f"  완료 ({time.perf_counter()-t0:.1f}s)")

In [ ]:
# ── 단일 질문 추론 ────────────────────────────────────────────────────────
QUESTION = "이 비디오에서 무엇이 일어나고 있나요? 구체적으로 설명해 주세요."
MAX_NEW_TOKENS = 128

print(f"질문: {QUESTION}\n")

# AutoGaze ON
_sync(); t0 = time.perf_counter()
ans_ag = runner_ag.run(frames, QUESTION, max_new_tokens=MAX_NEW_TOKENS)
_sync(); t_ag_full = time.perf_counter() - t0

# AutoGaze OFF
_sync(); t0 = time.perf_counter()
ans_base = runner_base.run(frames, QUESTION, max_new_tokens=MAX_NEW_TOKENS)
_sync(); t_base_full = time.perf_counter() - t0

print(f"AutoGaze ON  ({t_ag_full:.2f}s): {ans_ag[:120]}")
print()
print(f"AutoGaze OFF ({t_base_full:.2f}s): {ans_base[:120]}")
print()
saving = 100 * (t_base_full - t_ag_full) / t_base_full if t_base_full > 0 else 0
print(f"시간 절감: {saving:.1f}%")

---
## 5. Qwen2.5-VL — Hook vs Full 통합

In [ ]:
# ── 두 통합 방식 비교 ─────────────────────────────────────────────────────
print("""
Qwen2.5-VL AutoGaze 통합 방식 비교
─────────────────────────────────────────────────────────────────
  qwen25vl       zero-shot hook
    • 모델 수정 없음 (register_forward_hook)
    • 모든 temporal chunk에 동일한 mean gaze map 적용
    • 가장 빠른 적용 (zero-shot drop-in)

  qwen25vl_full  class monkey-patch
    • model.visual.__class__ = AutoGazeQwen25VisionTransformer
    • 각 temporal chunk (2프레임 단위)별로 다른 gaze map
    • 가중치 복사 없음 — forward() 동작만 오버라이드
    • INTEGRATION.md §Step 1 방식 (Qwen ViT 전용)

  핵심 차이:
    Qwen2.5-VL 비주얼 인코더는 cross-temporal attention이 없음
    (cu_seqlens로 temporal chunk 경계 분리)
    → block-causal 어텐션 마스크 불필요
    → zeroing 후 window reordering 이전 적용
─────────────────────────────────────────────────────────────────
""")

In [ ]:
# ── Qwen 로드 (주석 해제 후 실행) ────────────────────────────────────────
# 주의: transformers >= 4.45 필요, ~16 GB VRAM

LOAD_QWEN = False  # True로 변경하면 로드 시작

if LOAD_QWEN:
    print("[1/2] Qwen2.5-VL hook 방식 로드 중...")
    t0 = time.perf_counter()
    runner_qwen_hook = load_runner(
        mllm          = "qwen25vl",
        model_path    = QWEN_PATH,
        autogaze_path = str(AG_PATH),
        gazing_ratio  = RATIO,
        # integration="hook" is the default
    )
    print(f"  완료 ({time.perf_counter()-t0:.1f}s)")

    print("[2/2] Qwen2.5-VL full 통합 로드 중...")
    t0 = time.perf_counter()
    runner_qwen_full = load_runner(
        mllm          = "qwen25vl_full",
        model_path    = QWEN_PATH,
        autogaze_path = str(AG_PATH),
        gazing_ratio  = RATIO,
        # integration="full" is the default for qwen25vl_full
    )
    print(f"  완료 ({time.perf_counter()-t0:.1f}s)")

    # 추론
    ans_hook = runner_qwen_hook.run(frames, QUESTION, max_new_tokens=128)
    ans_full = runner_qwen_full.run(frames, QUESTION, max_new_tokens=128)

    print(f"\nhook  방식: {ans_hook[:100]}")
    print(f"full  방식: {ans_full[:100]}")
else:
    print("LOAD_QWEN = False — 실행하려면 True로 변경하세요.")

---
## 6. V-JEPA2 — 특징 추출 & LLM 추론

V-JEPA2는 세 가지 방식으로 사용할 수 있습니다.

| 키 | 역할 | MCQ 가능 |
|---|---|---|
| `vjepa2` | 특징 추출 (zero-shot hook) | ✗ |
| `vjepa2_full` | 특징 추출 (per-frame AutoGaze) | ✗ |
| `vjepa2_llm` | ViT + Projector + LLM 전체 파이프라인 | ✓ |

`vjepa2_llm`은 **projector 학습 필요** — `projector_path=None`이면 랜덤 초기화 (출력 무의미).

In [ ]:
print("""
V-JEPA2 파이프라인 구조 (vjepa2_llm)
─────────────────────────────────────────────────────────────────
  비디오 프레임 (T frames)
       ↓  AutoGaze  →  per-temporal-group gaze mask
  V-JEPA2 ViT-L  →  (B, T_p×H_p×W_p, 1024)
       ↓  VJEPA2Projector
          temporal mean pool  →  (B, T_p, 1024)
          LayerNorm + MLP     →  (B, T_p, lm_hidden)
       ↓  LLM (causal, e.g. Qwen2.5-7B)
          inputs_embeds = [video_tokens | text_tokens]
  MCQ 답변 생성

  T_p = num_frames / tubelet_size  (기본: 16 / 2 = 8 video tokens)

  Projector 학습:
    V-JEPA2 동결 + LLM 동결, projector 만 fine-tune
    projector.save_pretrained("weights/vjepa2_projector/")
─────────────────────────────────────────────────────────────────
""")

In [ ]:
# ── V-JEPA2 특징 추출 — vjepa2_full (주석 해제 후 실행) ──────────────────
# 주의: transformers >= 4.53 필요, ~4 GB VRAM

LOAD_VJEPA2 = False  # True로 변경하면 로드 시작

if LOAD_VJEPA2:
    print("V-JEPA2 + AutoGaze 로드 중...")
    t0 = time.perf_counter()
    runner_vjepa2 = load_runner(
        mllm          = "vjepa2_full",
        model_path    = VJEPA2_PATH,
        autogaze_path = str(AG_PATH),
        gazing_ratio  = RATIO,
    )
    print(f"  완료 ({time.perf_counter()-t0:.1f}s)")

    _sync(); t0 = time.perf_counter()
    features = runner_vjepa2.encode_video(frames)   # (1, N, C)
    _sync(); t_enc = time.perf_counter() - t0

    print(f"\n특징 텐서 shape : {features.shape}")
    print(f"인코딩 시간      : {t_enc*1000:.1f} ms")
    print(f"  N = T_p × H_p × W_p = {features.shape[1]}")
    print(f"  (e.g. 8 × 16 × 16 = 2048 for 16 frames)")
else:
    print("LOAD_VJEPA2 = False — 실행하려면 True로 변경하세요.")

In [ ]:
# ── V-JEPA2 LLM 추론 — vjepa2_llm (주석 해제 후 실행) ──────────────────
# 주의: V-JEPA2 + LLM 동시 로드, ~10-16 GB VRAM 필요
# projector_path=None 이면 랜덤 초기화 — 의미있는 답변 위해서는 학습 필요

LOAD_VJEPA2_LLM = False  # True로 변경하면 로드 시작

QUESTION_LLM = "What is the main activity shown in the video? A) Cooking B) Running C) Reading D) Swimming"

if LOAD_VJEPA2_LLM:
    import torch
    print("V-JEPA2 LLM runner 로드 중...")
    t0 = time.perf_counter()
    runner_vjepa2_llm = load_runner(
        mllm           = "vjepa2_llm",
        model_path     = VJEPA2_PATH,
        autogaze_path  = str(AG_PATH),
        gazing_ratio   = RATIO,
        lm_path        = LM_PATH,
        projector_path = PROJECTOR_PATH,   # None = random init (학습 후 경로 지정)
    )
    print(f"  완료 ({time.perf_counter()-t0:.1f}s)")

    # 추론
    print(f"\n질문: {QUESTION_LLM[:80]}")
    t0 = time.perf_counter()
    answer_ag_on  = runner_vjepa2_llm.run(frames, QUESTION_LLM)
    answer_ag_off = runner_vjepa2_llm.run(frames, QUESTION_LLM)  # ratio=0 은 별도 runner 필요
    elapsed = time.perf_counter() - t0

    print(f"\nAutoGaze ON  : {answer_ag_on}")
    print(f"소요 시간     : {elapsed*1000:.0f} ms")
    print()
    if PROJECTOR_PATH is None:
        print("⚠  projector_path=None — 랜덤 초기화 projector 사용 중")
        print("   의미있는 답변을 위해 projector 학습 후 경로를 지정하세요:")
        print("   PROJECTOR_PATH = 'weights/vjepa2_projector'")
else:
    print("LOAD_VJEPA2_LLM = False — 실행하려면 True로 변경하세요.")
    print()
    print("CLI 사용 예시:")
    print("  python autogaze/infer_full.py assets/example_input.mp4 \\")
    print("      --mllm vjepa2_llm \\")
    print("      --model-path facebook/vjepa2-vitl-fpc64-256 \\")
    print("      --autogaze-path weights/AutoGaze \\")
    print("      --lm-path Qwen/Qwen2.5-7B-Instruct \\")
    print("      --projector-path weights/vjepa2_projector \\")
    print("      --prompt 'What is happening? A) ... B) ... C) ... D) ...'"  )

---
## 7. Gazing Ratio Sweep

In [ ]:
# ── NVILA ratio sweep (로드된 runner 재사용) ──────────────────────────────
# 사전 요건: 섹션 4에서 runner_ag 로드 완료

RATIOS = [0.25, 0.50, 0.75, 1.0]
SWEEP_Q = "이 비디오에서 무엇이 일어나고 있나요?"

sweep_results = []   # (ratio, answer, elapsed_s, gaze_s)

print(f"질문: {SWEEP_Q}\n")
for r in RATIOS:
    # r=1.0 → AutoGaze OFF (selector 임시 해제)
    if r >= 1.0:
        _orig_sel = runner_ag.selector
        runner_ag.selector = None
        _sync(); t0 = time.perf_counter()
        ans = runner_ag.run(frames, SWEEP_Q, max_new_tokens=64)
        _sync(); t = time.perf_counter() - t0
        runner_ag.selector = _orig_sel
        gaze_s = 0.0
    else:
        runner_ag.selector.gazing_ratio = r
        runner_ag.gazing_ratio          = r
        _sync(); t0 = time.perf_counter()
        ans = runner_ag.run(frames, SWEEP_Q, max_new_tokens=64)
        _sync(); t = time.perf_counter() - t0
        gaze_s = 0.0  # (상세 AutoGaze 시간은 test_nvila.py 훅 방식으로 측정)

    sweep_results.append((r, ans, t, gaze_s))
    print(f"  ratio={r:.2f}  {t:.2f}s  →  {ans[:80]}")

# ratio 복원
runner_ag.selector.gazing_ratio = RATIO
runner_ag.gazing_ratio          = RATIO

In [ ]:
# ── Sweep 결과 시각화 ──────────────────────────────────────────────────────
ratios_  = [r for r, *_ in sweep_results]
times_   = [t for _, _, t, _ in sweep_results]
tokens_  = [int(r * 196) for r in ratios_]   # 14×14 = 196 patches

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 왼쪽: ratio vs 전체 시간
axes[0].plot([r*100 for r in ratios_], times_, "o-", color="#2196F3", lw=2.5, ms=10)
axes[0].axhline(times_[-1], color="gray", lw=1.5, ls="--", label="Baseline (r=1.0)")
for r, t in zip(ratios_, times_):
    axes[0].annotate(f"{t:.2f}s", (r*100, t), textcoords="offset points",
                     xytext=(0, 10), ha="center", fontsize=10)
axes[0].set_xlabel("Gazing Ratio (%)")
axes[0].set_ylabel("전체 추론 시간 (s)")
axes[0].set_title("Ratio vs 추론 시간", fontsize=12)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# 오른쪽: 처리 토큰 수 vs 시간 (파레토)
colors = ["#F44336" if r < 0.5 else "#FF9800" if r < 0.75 else "#4CAF50" for r in ratios_]
for r, tok, t, c in zip(ratios_, tokens_, times_, colors):
    axes[1].scatter(tok, t, color=c, s=180, zorder=5)
    axes[1].annotate(f"r={r}", (tok, t), textcoords="offset points", xytext=(8, 0), fontsize=9)
axes[1].plot(tokens_, times_, "--", color="gray", alpha=0.5, lw=1)
axes[1].set_xlabel("처리 패치 수 (196 = 100%)")
axes[1].set_ylabel("전체 추론 시간 (s)")
axes[1].set_title("토큰 절감 vs 속도 (파레토)", fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 8. 타이밍 분석

In [ ]:
# ── AutoGaze 단독 시간 측정 ───────────────────────────────────────────────
# runner._run_autogaze()를 직접 타이밍 래핑

def timed_run(runner, frames, question, max_new_tokens=64):
    gaze_time = [0.0]
    if hasattr(runner, '_run_autogaze') and runner.selector is not None:
        orig_ag = runner._run_autogaze
        def _timed(frm):
            _sync(); t0 = time.perf_counter()
            result = orig_ag(frm)
            _sync(); gaze_time[0] += time.perf_counter() - t0
            return result
        runner._run_autogaze = _timed
        _sync(); t0 = time.perf_counter()
        answer = runner.run(frames, question, max_new_tokens=max_new_tokens)
        _sync(); total = time.perf_counter() - t0
        runner._run_autogaze = orig_ag
    else:
        _sync(); t0 = time.perf_counter()
        answer = runner.run(frames, question, max_new_tokens=max_new_tokens)
        _sync(); total = time.perf_counter() - t0
    return answer, total, gaze_time[0]

print("타이밍 측정 중 (N=3 반복)...")
N_REPEAT = 3
lat_ag, lat_base = [], []
g_ag = []

for i in range(N_REPEAT):
    _, t_a, g_a = timed_run(runner_ag,   frames, QUESTION)
    _, t_b, _   = timed_run(runner_base, frames, QUESTION)
    lat_ag.append(t_a * 1000)
    lat_base.append(t_b * 1000)
    g_ag.append(g_a * 1000)
    print(f"  [{i+1}/{N_REPEAT}]  AG={t_a*1000:.0f}ms (gaze={g_a*1000:.0f}ms)  Base={t_b*1000:.0f}ms")

print(f"\n중앙값  AG  : {np.median(lat_ag):.1f} ms  (AutoGaze: {np.median(g_ag):.1f} ms)")
print(f"중앙값 Base : {np.median(lat_base):.1f} ms")
print(f"절감률      : {100*(np.median(lat_base)-np.median(lat_ag))/np.median(lat_base):.1f}%")

In [ ]:
# ── 타이밍 시각화 ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

x = np.arange(N_REPEAT)
# 왼쪽: 반복별 지연 시간
axes[0].plot(x, lat_ag,   "o-", color="#2196F3", lw=2, ms=8, label=f"AutoGaze ON (r={RATIO})")
axes[0].plot(x, lat_base, "s--", color="#607D8B", lw=2, ms=8, label="AutoGaze OFF")
axes[0].set_xlabel("반복 인덱스")
axes[0].set_ylabel("지연 시간 (ms)")
axes[0].set_title("반복별 전체 추론 시간", fontsize=12)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# 오른쪽: AG 시간 분해 (스택 바)
med_ag_total    = np.median(lat_ag)
med_ag_gaze     = np.median(g_ag)
med_ag_mllm     = med_ag_total - med_ag_gaze
med_base_total  = np.median(lat_base)

categories = ["AutoGaze ON", "AutoGaze OFF"]
ag_components  = [med_ag_gaze, med_ag_mllm]
base_total     = [0, med_base_total]

bars1 = axes[1].bar([0], [med_ag_gaze],  color="#FF9800", label="AutoGaze", alpha=0.9)
bars2 = axes[1].bar([0], [med_ag_mllm],  bottom=[med_ag_gaze], color="#2196F3", label="ViT+LLM", alpha=0.9)
bars3 = axes[1].bar([1], [med_base_total], color="#607D8B", label="Baseline", alpha=0.9)

axes[1].text(0, med_ag_gaze/2,                    f"{med_ag_gaze:.0f}ms",   ha="center", va="center", fontsize=10, color="white", fontweight="bold")
axes[1].text(0, med_ag_gaze + med_ag_mllm/2,      f"{med_ag_mllm:.0f}ms",  ha="center", va="center", fontsize=10, color="white", fontweight="bold")
axes[1].text(1, med_base_total/2,                  f"{med_base_total:.0f}ms",ha="center", va="center", fontsize=10, color="white", fontweight="bold")

axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(categories, fontsize=11)
axes[1].set_ylabel("지연 시간 (ms)")
axes[1].set_title("추론 시간 분해", fontsize=12)
axes[1].legend(fontsize=9)
axes[1].grid(axis="y", alpha=0.3)

saving = 100 * (med_base_total - med_ag_total) / med_base_total
fig.text(0.5, -0.03, f"절감률 {saving:.1f}% (AutoGaze ON vs OFF)",
         ha="center", fontsize=10, color="#555", style="italic")

plt.tight_layout()
plt.show()

---
## 9. 전체 CLI 가이드

In [ ]:
guide = """
╔══════════════════════════════════════════════════════════════════════════╗
║               추론 스크립트 사용법 요약                                 ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  ── infer.py (AutoGaze 전용 gaze 추출) ──────────────────────────────── ║
║  python -m autogaze.infer video.mp4                                      ║
║  python -m autogaze.infer video.mp4 --output-format video                ║
║  python -m autogaze.infer video.mp4 --all-frames --output-format video   ║
║                                                                          ║
║  ── infer_full.py (AutoGaze + ViT + MLLM) ───────────────────────────── ║
║  # NVILA 기본 (AutoGaze ON)                                              ║
║  python autogaze/infer_full.py video.mp4 --mllm nvila                    ║
║                                                                          ║
║  # AutoGaze ON/OFF 비교                                                  ║
║  python autogaze/infer_full.py video.mp4 --compare-autogaze              ║
║                                                                          ║
║  # Qwen2.5-VL, full ViT 통합                                             ║
║  python autogaze/infer_full.py video.mp4 \\                              ║
║      --mllm qwen25vl_full \\                                             ║
║      --model-path Qwen/Qwen2.5-VL-7B-Instruct                            ║
║                                                                          ║
║  # V-JEPA2 특징 추출 (projector+LLM 없이)                               ║
║  python autogaze/infer_full.py video.mp4 \\                              ║
║      --mllm vjepa2_full \\                                               ║
║      --model-path facebook/vjepa2-vitl-fpc64-256                         ║
║                                                                          ║
║  # V-JEPA2 + projector + LLM (MCQ 추론)                                 ║
║  python autogaze/infer_full.py video.mp4 \\                              ║
║      --mllm vjepa2_llm \\                                                ║
║      --model-path facebook/vjepa2-vitl-fpc64-256 \\                      ║
║      --lm-path Qwen/Qwen2.5-7B-Instruct \\                               ║
║      --projector-path weights/vjepa2_projector \\                        ║
║      --prompt "What is shown? A) ... B) ..."                             ║
║                                                                          ║
║  # Gaze 시각화 저장                                                      ║
║  python autogaze/infer_full.py video.mp4 --save-gaze                     ║
║                                                                          ║
║  ── 벤치마크 평가 ────────────────────────────────────────────────────── ║
║  bash scripts/run_benchmarks.sh --tasks videomme,mvbench                 ║
║  bash scripts/run_benchmarks.sh --mllm qwen25vl_full \\                  ║
║       --model-path Qwen/Qwen2.5-VL-7B-Instruct                          ║
║  bash scripts/run_benchmarks.sh --mllm vjepa2_llm \\                     ║
║       --model-path facebook/vjepa2-vitl-fpc64-256 \\                     ║
║       --lm-path Qwen/Qwen2.5-7B-Instruct \\                              ║
║       --projector-path weights/vjepa2_projector                          ║
║                                                                          ║
║  자세한 내용: docs/inference_guide.md                                    ║
╚══════════════════════════════════════════════════════════════════════════╝
"""
print(guide)